# Fine-tuning LoRA d'un modèle médical — TechCorp (mission R&D)

**Filière : IA — Mission expérimentale (pas pour la production).**

Ce notebook réalise un fine-tuning **QLoRA (4-bit)** de `microsoft/Phi-3.5-mini-instruct` sur le
dataset `ruslanmv/ai-medical-chatbot`, avec journalisation des métriques (loss, epochs).

Environnement cible : **Google Colab** (GPU T4/A100). Runtime → Modifier le type d'exécution → GPU.

> ⚠️ Modèle **expérimental**. Un LLM médical ne remplace jamais un avis médical humain et ne doit
> pas être déployé en production sans validation clinique (voir `medical_project/Readme.md`).

## 1. Installation des dépendances

In [ ]:
!pip -q install "transformers>=4.45" "peft>=0.12" "accelerate>=0.34" \
    "bitsandbytes>=0.43" "datasets>=2.20" "trl>=0.9" sentencepiece

## 2. Vérification du GPU

In [ ]:
import torch
print('CUDA disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))

## 3. Chargement du modèle de base en 4-bit (QLoRA)

La quantification 4-bit (NF4) permet de fine-tuner un modèle 3.8B sur un GPU Colab standard.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = 'microsoft/Phi-3.5-mini-instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=False,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

## 4. Configuration LoRA

On adapte uniquement les projections d'attention et MLP (≈1 % des paramètres entraînés).

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['qkv_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Dataset médical

On charge `ruslanmv/ai-medical-chatbot` et on le met au format chat de Phi-3.
Pour un run de démonstration Colab, on échantillonne (ajustez `SAMPLE`).

In [ ]:
from datasets import load_dataset

SAMPLE = 800  # run de démonstration (~10-15 min sur T4). Augmentez (ex. 3000) ou None pour un entraînement plus complet.
raw = load_dataset('ruslanmv/ai-medical-chatbot', split='train')
if SAMPLE:
    raw = raw.select(range(min(SAMPLE, len(raw))))
print(raw)
print(raw.column_names)

In [ ]:
def to_text(row):
    patient = str(row.get('Patient', '')).strip()
    doctor = str(row.get('Doctor', '')).strip()
    text = (f'<|user|>\n{patient}<|end|>\n'
            f'<|assistant|>\n{doctor}<|end|>')
    return {'text': text}

dataset = raw.map(to_text, remove_columns=raw.column_names)
dataset = dataset.filter(lambda r: len(r['text']) > 40)
print('Exemples après nettoyage :', len(dataset))
print(dataset[0]['text'][:300])

## 6. Entraînement (avec journalisation des métriques)

In [ ]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling, Trainer

def tokenize(batch):
    out = tokenizer(batch['text'], truncation=True, max_length=512, padding='max_length')
    out['labels'] = out['input_ids'].copy()
    return out

tokenized = dataset.map(tokenize, batched=True, remove_columns=['text'])

args = TrainingArguments(
    output_dir='phi35_medical_lora',
    num_train_epochs=1,   # 1 epoch suffit pour la démonstration ; augmentez pour un modèle plus abouti
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=10,
    save_strategy='epoch',
    fp16=True,
    optim='paged_adamw_8bit',
    report_to='none',
)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = Trainer(model=model, args=args, train_dataset=tokenized, data_collator=collator)
train_result = trainer.train()

## 7. Métriques d'entraînement

In [ ]:
import pandas as pd

history = pd.DataFrame(trainer.state.log_history)
loss_curve = history.dropna(subset=['loss'])[['epoch', 'step', 'loss']]
print(loss_curve.to_string(index=False))
print('\nMétriques finales :', train_result.metrics)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7,4))
plt.plot(loss_curve['step'], loss_curve['loss'])
plt.xlabel('step'); plt.ylabel('training loss'); plt.title('Courbe de loss — LoRA médical')
plt.grid(True); plt.tight_layout(); plt.show()

## 8. Sauvegarde de l'adaptateur

In [ ]:
model.save_pretrained('phi35_medical_lora')
tokenizer.save_pretrained('phi35_medical_lora')
print('Adaptateur LoRA sauvegardé dans phi35_medical_lora/')
# Optionnel : monter Google Drive et y copier le dossier pour partage.

## 9. Test conversationnel

In [ ]:
def generate(prompt, max_new_tokens=200):
    text = f'<|user|>\n{prompt}<|end|>\n<|assistant|>\n'
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    model.eval()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=0.3, top_p=0.9, do_sample=True,
                             repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

for q in ['What are common symptoms of dehydration?',
          'Is a persistent headache with fever something to worry about?']:
    print('Q:', q)
    print('A:', generate(q).strip())
    print('-'*60)

## 10. Métriques à reporter (livrable)

À copier dans `rendu/ia/evaluation_phi35_financial.md` (section médicale) après le run :

| Métrique | Valeur |
|---|---|
| Modèle de base | microsoft/Phi-3.5-mini-instruct |
| Méthode | QLoRA 4-bit (NF4), r=16, alpha=32 |
| Échantillons | (voir SAMPLE) |
| Epochs | 3 |
| Loss initiale → finale | *(remplir depuis la section 7)* |
| Durée d'entraînement | *(train_result.metrics['train_runtime'])* |
| Lien Colab | *(Partager → copier le lien)* |

> Rappel sécurité (cohérent avec l'audit CYBER) : ce dataset et ce modèle doivent passer le même
> contrôle anti-empoisonnement (`rendu/data/analyze_datasets.py`) avant tout usage élargi.